# Working with Randomness and Probability — Extended Walk-Through

Based on *Chapter 4* of **Applying Math with Python (2nd Edition)** by Sam Morley, with additions, deeper explanations, visualisations, and exercises.

**Contents**
1. Selecting items at random
2. Generating random data (uniform)
3. Changing the random number generator
4. Generating normally distributed random numbers
5. Working with random processes (Poisson)
6. Analysing conversion rates with Bayesian techniques
7. Estimating parameters with Monte Carlo / MCMC
8. Bonus — Monte Carlo estimation of π
9. Exercises

In [ ]:
# ── imports & global config ───────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
from scipy.special import factorial
from numpy.random import default_rng

plt.rcParams.update({
    "figure.figsize": (8, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

SEED = 12345
rng = default_rng(SEED)
print("NumPy", np.__version__, "| SciPy", sp.__version__)

---
## 1. Selecting Items at Random

Key ideas: **sample space**, **event**, **probability**, **PRNG**, **seed**.

We use `np.random.default_rng(seed)` to create a reproducible generator and the `.choice()` method to draw from a weighted discrete distribution.

In [ ]:
data = np.arange(15)
probabilities = np.array([
    0.30, 0.20, 0.10, 0.05, 0.05,
    0.05, 0.05, 0.025, 0.025, 0.025,
    0.025, 0.025, 0.025, 0.025, 0.025
])
assert round(sum(probabilities), 10) == 1.0, "Probabilities must sum to 1"

selected = rng.choice(data, p=probabilities, replace=True)
print("Single selection:", selected)

selected_array = rng.choice(
    data, p=probabilities, replace=True, size=(5, 5)
)
print("5×5 selection:\n", selected_array)

In [ ]:
# ── extended: empirical frequency vs theoretical probability ─────
N = 100_000
big_sample = rng.choice(data, p=probabilities, replace=True, size=N)
empirical_freq = np.bincount(big_sample, minlength=15) / N

fig, ax = plt.subplots(figsize=(9, 4))
width = 0.4
ax.bar(data - width/2, probabilities, width, label="Theoretical", alpha=0.7)
ax.bar(data + width/2, empirical_freq, width, label=f"Empirical (N={N:,})", alpha=0.7)
ax.set_xlabel("Item"); ax.set_ylabel("Probability")
ax.set_title("Theoretical vs Empirical Probabilities")
ax.legend()
plt.tight_layout(); plt.show()

> **Law of Large Numbers** — as *N* grows the empirical frequencies converge to the theoretical probabilities. The bar plot above should be nearly indistinguishable at large *N*.

---
## 2. Generating Random Data (Uniform)

`rng.random()` → floats in `[0, 1)`  
`rng.integers(low, high, endpoint=True)` → integers in `[low, high]`

We visualise how histograms become flatter (approaching a true uniform PDF) as sample size increases.

In [ ]:
random_floats = rng.random(size=(5, 5))
print("5×5 floats:\n", random_floats)

random_ints = rng.integers(1, 20, endpoint=True, size=10)
print("10 ints in [1,20]:", random_ints)

In [ ]:
# ── extended: convergence to uniformity at different N ───────────
sizes = [100, 1_000, 10_000, 100_000]
fig, axes = plt.subplots(1, len(sizes), figsize=(16, 4), sharey=True)
for ax, n in zip(axes, sizes):
    d = rng.random(size=n)
    ax.hist(d, bins=30, color="tab:blue", alpha=0.6)
    ax.set_title(f"N = {n:,}")
    ax.set_xlabel("Value")
axes[0].set_ylabel("Count")
fig.suptitle("Convergence of Uniform Samples to Flat Histogram", y=1.03)
plt.tight_layout(); plt.show()

---
## 3. Changing the Random Number Generator

NumPy offers several **BitGenerators**:

| Generator  | Speed  | Quality  | Notes                          |
|-----------|--------|----------|--------------------------------|
| PCG64     | Fast   | High     | Default                        |
| MT19937   | Slow   | Moderate | Same algo as Python's `random` |
| Philox    | Slow   | Very high| Counter-based                  |
| SFC64     | Fast   | Reasonable| Short period                   |

`SeedSequence` lets you spawn independent streams reproducibly.

In [ ]:
from numpy import random

seed_seq = random.SeedSequence()
print("Entropy:", seed_seq.entropy)

bit_gen = random.MT19937(seed_seq)
rng_mt = random.Generator(bit_gen)
print("MT19937 sample:", rng_mt.random(3))

# ── extended: compare generators visually ──────────────────────
configs = {
    "PCG64":  random.PCG64(SEED),
    "MT19937": random.MT19937(SEED),
    "Philox":  random.Philox(SEED),
    "SFC64":   random.SFC64(SEED),
}
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharex=True, sharey=True)
for ax, (name, bg) in zip(axes, configs.items()):
    g = random.Generator(bg)
    vals = g.random(5000)
    ax.scatter(vals[:-1], vals[1:], s=1, alpha=0.3)
    ax.set_title(name)
    ax.set_aspect("equal")
fig.suptitle("2-D Scatter of Consecutive Uniform Draws (lag-1)", y=1.03)
plt.tight_layout(); plt.show()

---
## 4. Generating Normally Distributed Random Numbers

Normal PDF:  
$$f(x) = \frac{1}{\sigma\sqrt{2\pi}}\,\exp\!\left(-\frac{1}{2}\left(\frac{x-\mu}{\sigma}\right)^2\right)$$

We draw samples and overlay the theoretical curve to verify the fit.

In [ ]:
mu, sigma = 5.0, 3.0
rands = rng.normal(loc=mu, scale=sigma, size=10_000)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(rands, bins=40, density=True, color="tab:blue", alpha=0.6, label="Sample")

x_range = np.linspace(-5, 15, 500)
pdf = np.exp(-0.5*((x_range - mu)/sigma)**2) / (sigma * np.sqrt(2*np.pi))
ax.plot(x_range, pdf, "k--", lw=2, label="Theoretical PDF")
ax.set_title(f"Normal Distribution (μ={mu}, σ={sigma})")
ax.set_xlabel("Value"); ax.set_ylabel("Density")
ax.legend()
plt.tight_layout(); plt.show()

print(f"Sample mean  = {rands.mean():.3f} (true {mu})")
print(f"Sample std   = {rands.std():.3f} (true {sigma})")
print(f"Skewness    = {sp.stats.skew(rands):.4f}")
print(f"Kurtosis    = {sp.stats.kurtosis(rands):.4f}  (normal → 0)")

In [ ]:
# ── extended: sample other distributions ───────────────────────────
dists = {
    "Exponential":  rng.exponential(scale=2.0, size=10_000),
    "Beta(2,5)":   rng.beta(2, 5, size=10_000),
    "Gamma(3,2)":  rng.gamma(3, 2, size=10_000),
    "Lognormal":   rng.lognormal(mean=0, sigma=1, size=10_000),
}
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (name, data) in zip(axes.ravel(), dists.items()):
    ax.hist(data, bins=50, density=True, color="tab:purple", alpha=0.6)
    ax.set_title(name)
    ax.set_xlabel("Value"); ax.set_ylabel("Density")
fig.suptitle("Samples from Various Continuous Distributions", y=1.01)
plt.tight_layout(); plt.show()

---
## 5. Working with Random Processes (Poisson Process)

A **Poisson process** with rate $\lambda$ models the count of events over time. Inter-arrival times are exponentially distributed with scale $1/\lambda$.

$$P(N(t)=n) = \frac{(\lambda t)^n}{n!}e^{-\lambda t}$$

In [ ]:
rate = 4.0
inter_arrival = rng.exponential(scale=1/rate, size=50)
arrivals = np.add.accumulate(inter_arrival)
count = np.arange(50)

fig1, ax1 = plt.subplots(figsize=(9, 4))
ax1.step(arrivals, count, where="post", lw=2)
ax1.set_xlabel("Time"); ax1.set_ylabel("Number of arrivals")
ax1.set_title(f"Poisson Process Arrivals (λ={rate})")
plt.tight_layout(); plt.show()

In [ ]:
N_vals = np.arange(0, 20)

def poisson_pmf(events, time=1, param=rate):
    return ((param * time)**events / factorial(events)) * np.exp(-param * time)

estimated_scale = np.mean(inter_arrival)
estimated_rate  = 1.0 / estimated_scale

fig2, ax2 = plt.subplots(figsize=(9, 4))
ax2.plot(N_vals, poisson_pmf(N_vals), "k-", lw=2, label=f"True (λ={rate})")
ax2.plot(N_vals, poisson_pmf(N_vals, param=estimated_rate), "k--",
         lw=2, label=f"Estimated (λ̂={estimated_rate:.3f})")
ax2.set_xlabel("Number of arrivals in 1 time unit")
ax2.set_ylabel("Probability")
ax2.set_title("Poisson PMF — True vs Estimated")
ax2.legend()
plt.tight_layout(); plt.show()

print(f"True rate = {rate}  |  Estimated rate = {estimated_rate:.4f}")

In [ ]:
# ── extended: 2-state homogeneous Markov chain ────────────────────
T = np.array([[0.4, 0.6],
              [0.8, 0.2]])

def simulate_markov(T, start=0, steps=200):
    n_states = T.shape[0]
    state = start
    history = [state]
    for _ in range(steps):
        state = rng.choice(n_states, p=T[state])
        history.append(state)
    return np.array(history)

chain = simulate_markov(T, start=0, steps=500)
fig, (axA, axB) = plt.subplots(1, 2, figsize=(14, 4))
axA.plot(chain[:100], drawstyle="steps-post")
axA.set_title("First 100 Steps of Markov Chain")
axA.set_xlabel("Step"); axA.set_ylabel("State")

# stationary distribution via eigen-decomposition
eigvals, eigvecs = np.linalg.eig(T.T)
stationary = eigvecs[:, np.isclose(eigvals, 1)].real.flatten()
stationary /= stationary.sum()
empirical = np.bincount(chain, minlength=2) / len(chain)
axB.bar(["A (true)", "A (emp)", "B (true)", "B (emp)"],
        [stationary[0], empirical[0], stationary[1], empirical[1]],
        color=["tab:blue","tab:cyan","tab:orange","tab:red"], alpha=0.7)
axB.set_title("Stationary vs Empirical Distribution")
axB.set_ylabel("Probability")
plt.tight_layout(); plt.show()
print(f"Stationary dist = {stationary.round(4)}  |  Empirical = {empirical.round(4)}")

---
## 6. Analysing Conversion Rates with Bayesian Techniques

We model a conversion rate with a **Beta prior** and update it with observed successes/failures (binomial likelihood → Beta is the conjugate prior).

Bayes' theorem (proportional form):
$$P(\theta | \text{data}) \propto P(\text{data} | \theta)\,P(\theta)$$

Posterior parameters: $\alpha' = \alpha + m$, $\beta' = \beta + n$.

In [ ]:
from scipy.stats import beta as beta_dist
beta_pdf = beta_dist.pdf

prior_alpha, prior_beta = 25, 75
observed_successes = 122
observed_failures  = 257

posterior_alpha = prior_alpha + observed_successes
posterior_beta  = prior_beta + observed_failures

prior_over_33, _    = sp.integrate.quad(beta_pdf, 0.33, 1, args=(prior_alpha, prior_beta))
posterior_over_33,_ = sp.integrate.quad(beta_pdf, 0.33, 1, args=(posterior_alpha, posterior_beta))

print(f"Prior   P(rate > 33%) = {prior_over_33:.4f}")
print(f"Posterior P(rate > 33%) = {posterior_over_33:.4f}")
print(f"Posterior mean       = {posterior_alpha/(posterior_alpha+posterior_beta):.4f}")
print(f"Posterior mode       = {(posterior_alpha-1)/(posterior_alpha+posterior_beta-2):.4f}")

In [ ]:
p = np.linspace(0, 1, 500)
prior_dist    = beta_pdf(p, prior_alpha, prior_beta)
posterior_dist = beta_pdf(p, posterior_alpha, posterior_beta)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(p, prior_dist, "k--", lw=2, label="Prior Beta(25,75)")
ax.plot(p, posterior_dist, "k-", lw=2, label=f"Posterior Beta({posterior_alpha},{posterior_beta})")
ax.axvline(0.33, color="red", ls=":", label="Threshold 33%")
ax.fill_between(p[p>=0.33], 0, posterior_dist[p>=0.33], alpha=0.15, color="green")
ax.set_xlabel("Success rate"); ax.set_ylabel("Density")
ax.set_title("Prior and Posterior Distributions")
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── extended: sequential Bayesian updating ───────────────────────
# Simulate data arriving in 5 batches and watch the posterior evolve
batches = [(30, 60), (20, 50), (25, 40), (35, 55), (12, 52)]
alpha, beta_param = prior_alpha, prior_beta
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(p, beta_pdf(p, alpha, beta_param), "k--", lw=1, alpha=0.4, label="Prior")
colors = plt.cm.viridis(np.linspace(0, 1, len(batches)))
for i, (succ, fail) in enumerate(batches):
    alpha += succ
beta_param += fail   # NOTE: align indentation with loop body
    # The line above is illustrative; corrected version below

# Re-run correctly
alpha, beta_param = prior_alpha, prior_beta
ax.clear()
ax.plot(p, beta_pdf(p, alpha, beta_param), "k--", lw=1, alpha=0.4, label="Prior")
for i, (succ, fail) in enumerate(batches):
    alpha += succ
    beta_param += fail
    ax.plot(p, beta_pdf(p, alpha, beta_param), color=colors[i], lw=2,
            label=f"After batch {i+1}: α={alpha},β={beta_param}")
ax.set_xlabel("Success rate"); ax.set_ylabel("Density")
ax.set_title("Sequential Bayesian Updating")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

---
## 7. Estimating Parameters with Monte Carlo / MCMC (PyMC)

We generate noisy data from a quadratic $y = ax^2 + bx + c$ and use **PyMC** with the **NUTS** sampler to recover the coefficients.

*(Install PyMC first: `pip install pymc`)*

In [ ]:
def underlying(x, params):
    return params[0]*x**2 + params[1]*x + params[2]

size = 100
true_params = [2, -7, 6]
x_vals = np.linspace(-5, 5, size)
raw_model = underlying(x_vals, true_params)
noise = rng.normal(loc=0.0, scale=10.0, size=size)
sample = raw_model + noise

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(x_vals, sample, color="k", alpha=0.5, s=20, label="Noisy sample")
ax.plot(x_vals, raw_model, "r--", lw=2, label="True model")
ax.set_title("Quadratic Data with Noise")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
import pymc as pm
import arviz as az

with pm.Model() as model:
    params = pm.Normal("params", mu=1, sigma=1, shape=3)
    y = underlying(x_vals, params)
    y_obs = pm.Normal("y_obs", mu=y, sigma=2, observed=sample)
    trace = pm.sample(cores=4, random_seed=SEED)

az.summary(trace, var_names=["params"])

In [ ]:
az.plot_posterior(trace, var_names=["params"], figsize=(14, 4))
plt.tight_layout(); plt.show()

estimated_params = trace.posterior["params"].mean(axis=(0,1)).to_numpy()
print("True params:     ", true_params)
print("Estimated params: ", estimated_params.round(3))

In [ ]:
estimated = underlying(x_vals, estimated_params)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(x_vals, raw_model, "k-", lw=2, label="True model")
ax.plot(x_vals, estimated, "k--", lw=2, label="Estimated model")
ax.scatter(x_vals, sample, color="grey", alpha=0.3, s=15, label="Data")
ax.set_title("True vs Estimated Model")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── extended: posterior predictive traces ─────────────────────────
with model:
    ppc = pm.sample_posterior_predictive(trace, var_names=["y_obs"],
                                          random_seed=SEED)

ppc_samples = ppc["posterior_predictive"]["y_obs"].values.reshape(-1, size)
fig, ax = plt.subplots(figsize=(9, 4))
for i in range(0, 200, 20):
    ax.plot(x_vals, ppc_samples[i], color="tab:blue", alpha=0.05)
ax.plot(x_vals, raw_model, "r-", lw=2, label="True")
ax.set_title("Posterior Predictive Samples")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.legend(["True"])
plt.tight_layout(); plt.show()

---
## 8. Bonus — Monte Carlo Estimation of π

Generate uniform points in `[-1,1]²`. The fraction inside the unit circle × 4 ≈ π.

In [ ]:
def estimate_pi(n_points=10_000, rng=None):
    if rng is None:
        rng = default_rng()
    pts = rng.uniform(-1, 1, size=(2, n_points))
    inside = pts[0]**2 + pts[1]**2 < 1
    return 4.0 * inside.sum() / n_points, pts, inside

pi_est, pts, inside = estimate_pi(20_000, rng=rng)
print(f"π estimate (20k pts): {pi_est:.5f}")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(pts[0, inside],  pts[1, inside],  s=1, color="tab:green", alpha=0.3, label="Inside")
ax.scatter(pts[0, ~inside], pts[1, ~inside], s=1, color="tab:red",   alpha=0.3, label="Outside")
theta = np.linspace(0, 2*np.pi, 200)
ax.plot(np.cos(theta), np.sin(theta), "k-", lw=2)
ax.set_aspect("equal"); ax.set_title(f"π ≈ {pi_est:.4f}")
ax.legend(markerscale=5)
plt.tight_layout(); plt.show()

# Average over 100 runs
from statistics import mean
results = [estimate_pi(rng=rng)[0] for _ in range(100)]
print(f"π estimate (avg of 100 runs): {mean(results):.5f}")

---
## 9. Exercises

1. **Dice simulation.** Simulate rolling two fair dice 100 000 times. Plot the distribution of the sum. What is the most probable sum?
2. **Birthday paradox.** Simulate rooms of *n* people (*n* = 10…60). For each *n*, run 10 000 trials and plot the probability that at least two people share a birthday.
3. **Confidence interval for the mean.** Draw 1 000 samples of size 50 from a normal distribution (μ=10, σ=5). For each sample compute the 95% CI for the mean. Plot the intervals and count how many contain μ=10.
4. **Change-point detection (Bayesian).** Extend the conversion-rate analysis: assume the rate may have changed midway. Use two Beta priors and compare posterior probabilities.
5. **Monte Carlo integration.** Estimate $\int_0^1 x^2\,dx$ using Monte Carlo sampling and compare with the analytical answer $1/3$.
6. **MCMC with different priors.** Re-run the PyMC model with Uniform(-20,20) priors instead of Normal(1,1). Does the posterior change significantly? Why or why not?